[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/scw634919-bfty/ecommerce-data-analytics-portfolio/blob/main/5-product-performance-dashboard-kpi/notebook/product_performance_dashboard_kpi.ipynb)

# Product Performance Dashboard (KPI)

Analyze product performance using key business metrics to better understand sales performance, product demand, and product trends.

**Business Goal:** Identify high-performing products, monitor KPI patterns, and support business decision-making through a clear product performance dashboard.


## 1. Import Libraries

Import libraries for data analysis, KPI tracking, and visualization.

In [1]:
import pandas as pd

# 1. Load data
df = pd.read_excel("Online Retail.xlsx")

## 2. Load Dataset

Load product and sales-related data and inspect the dataset structure.

In [2]:
# Data cleaning (aligned with the portfolio-standard cleaning used in Project 1)
df.columns = df.columns.str.lower()

# Remove records that cannot be attributed to a customer/product
df = df.dropna(subset=["customerid", "description"])

# Remove cancellations / returns and invalid prices
df = df[df["quantity"] > 0]
df = df[df["unitprice"] > 0]

# Parse dates and build the revenue column
df["invoicedate"] = pd.to_datetime(df["invoicedate"])
df["sales"] = df["quantity"] * df["unitprice"]

# Standardize column names for the KPI layer
df = df.rename(columns={
    "invoiceno": "order_id",
    "customerid": "customer_id",
    "invoicedate": "order_date",
    "unitprice": "unit_price",
    "stockcode": "product_id",
    "description": "product_name"
})

## 3. Data Cleaning

Clean missing values and prepare the dataset for KPI analysis.

In [3]:
# Keyword-based product categorization (consistent with Project 3's scheme)
def categorize(desc):
    desc = str(desc).lower()

    # Home decor (largest category in this dataset)
    if any(w in desc for w in [
        "candle", "holder", "lamp", "frame", "mirror", "vase", "clock",
        "heart", "flower", "light", "decor", "ornament", "door",
        "picture", "sign", "hanger", "shelf", "storage", "box", "drawer"
    ]):
        return "home decor"
    # Kitchen / dining
    elif any(w in desc for w in [
        "kitchen", "tea", "coffee", "mug", "cup", "plate", "bowl", "glass",
        "bottle", "jar", "tin", "cutlery", "spoon", "fork", "knife", "tray",
        "baking", "pan", "kettle", "teapot", "cake"
    ]):
        return "kitchen"
    # Bags
    elif any(w in desc for w in ["bag", "purse", "tote", "backpack"]):
        return "bags"
    # Gift / decorative
    elif any(w in desc for w in [
        "gift", "present", "ribbon", "card", "wrap", "wrapping",
        "party", "bunting", "decorative"
    ]):
        return "gift/decorative"
    # Seasonal
    elif any(w in desc for w in [
        "christmas", "xmas", "santa", "snow", "tree", "easter",
        "halloween", "valentine", "seasonal"
    ]):
        return "seasonal"
    # Sets / bundles
    elif any(w in desc for w in ["set", "kit", "pack", "bundle"]):
        return "sets"
    # Stationery
    elif any(w in desc for w in [
        "paper", "notebook", "note", "stationery", "pen", "pencil",
        "file", "folder", "sticker"
    ]):
        return "stationery"
    else:
        return "other"

df["category"] = df["product_name"].apply(categorize)

## 4. Exploratory Data Analysis (EDA)

Explore product performance, sales behavior, and business metrics.

In [4]:
# Monthly KPI (used for the dashboard's monthly trend line)
df["year_month"] = df["order_date"].dt.to_period("M").astype(str)

monthly_kpi = df.groupby("year_month").agg(
    total_revenue=("sales", "sum"),
    total_orders=("order_id", "nunique"),
    total_customers=("customer_id", "nunique"),
    units_sold=("quantity", "sum")
).reset_index()

monthly_kpi["average_order_value"] = (
    monthly_kpi["total_revenue"] / monthly_kpi["total_orders"]
)

print("Monthly KPI")
print(monthly_kpi.head())

# ------------------------------------------------------------------
# Overall KPI summary (single row) for the dashboard KPI cards.
#
# IMPORTANT: total_customers must be a DISTINCT count over the whole
# dataset. We must NOT sum the monthly `total_customers` column, because
# a customer who buys in multiple months would be counted once per month
# (double counting). The distinct count below is the correct total.
# ------------------------------------------------------------------
kpi_summary = pd.DataFrame([{
    "total_revenue": round(df["sales"].sum(), 2),
    "total_orders": df["order_id"].nunique(),
    "total_customers": df["customer_id"].nunique(),
    "units_sold": int(df["quantity"].sum()),
    "average_order_value": round(df["sales"].sum() / df["order_id"].nunique(), 2),
}])

print("\nOverall KPI Summary")
print(kpi_summary.to_string(index=False))

Monthly KPI
  year_month  total_revenue  total_orders  total_customers  units_sold  \
0    2010-12     572713.890          1400              885      312265   
1    2011-01     569445.040           987              741      349098   
2    2011-02     447137.350           997              758      265622   
3    2011-03     595500.760          1321              974      348503   
4    2011-04     469200.361          1149              856      292222   

   average_order_value  
0           409.081350  
1           576.945329  
2           448.482798  
3           450.795428  
4           408.355406  

Overall KPI Summary
 total_revenue  total_orders  total_customers  units_sold  average_order_value
     8911407.9         18532             4338     5167812               480.87


## 5. KPI Analysis

Calculate key performance indicators to evaluate product performance.

In [5]:
# Category KPI (revenue / orders / units by product category)
category_kpi = df.groupby("category").agg(
    revenue=("sales", "sum"),
    orders=("order_id", "nunique"),
    units_sold=("quantity", "sum")
).reset_index().sort_values("revenue", ascending=False)

print("Category KPI")
print(category_kpi)

# Category performance for the dashboard bar chart (excludes "other"
# for clearer business interpretation, adds SKU count and revenue share)
category_performance = (
    df[df["category"] != "other"]
    .groupby("category")
    .agg(
        total_units_sold=("quantity", "sum"),
        total_revenue=("sales", "sum"),
        sku_count=("product_id", "nunique"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
category_performance["revenue_pct"] = (
    category_performance["total_revenue"] / category_performance["total_revenue"].sum() * 100
).round(2)

print("\nCategory Performance (excl. other)")
print(category_performance)

Category KPI
          category      revenue  orders  units_sold
2       home decor  3666701.330   16547     2049771
3          kitchen  1871562.800   14059      877623
4            other  1458003.564   14058      669824
0             bags   744968.400    7894      457683
6             sets   420541.840    8617      344691
7       stationery   286178.240    4079      256468
1  gift/decorative   270192.910    6545      368711
5         seasonal   193258.820    3898      143041

Category Performance (excl. other)
          category  total_units_sold  total_revenue  sku_count  revenue_pct
2       home decor           2049771     3666701.33       1341        49.19
3          kitchen            877623     1871562.80        620        25.11
0             bags            457683      744968.40        155        10.00
5             sets            344691      420541.84        186         5.64
6       stationery            256468      286178.24        119         3.84
1  gift/decorative         

## 6. Dashboard Development

Build dashboard metrics and summarize product performance trends.

In [6]:
import os
os.makedirs("../outputs", exist_ok=True)

# Row-level cleaned dataset — use this as the Tableau source so the KPI
# cards can compute COUNTD(customer_id) / COUNTD(order_id) correctly.
cleaned_cols = [
    "order_id", "order_date", "year_month", "customer_id",
    "product_id", "product_name", "category",
    "quantity", "unit_price", "sales", "country"
]
df[cleaned_cols].to_csv("cleaned_kpi_data.csv", index=False)

# Aggregated tables for the dashboard
kpi_summary.to_csv("../outputs/kpi_summary.csv", index=False)
monthly_kpi.to_csv("../outputs/monthly_kpi.csv", index=False)
category_kpi.to_csv("../outputs/category_kpi.csv", index=False)
category_performance.to_csv("../outputs/category_performance.csv", index=False)

# Convenience copies next to the notebook
kpi_summary.to_csv("kpi_summary.csv", index=False)
monthly_kpi.to_csv("monthly_kpi.csv", index=False)
category_kpi.to_csv("category_kpi.csv", index=False)

print("Saved:")
print("  cleaned_kpi_data.csv -", f"{len(df):,}", "rows (Tableau source)")
print("  ../outputs/kpi_summary.csv - overall KPI totals")
print("  ../outputs/monthly_kpi.csv -", len(monthly_kpi), "months")
print("  ../outputs/category_kpi.csv -", len(category_kpi), "categories")
print("  ../outputs/category_performance.csv -", len(category_performance), "categories (excl. other)")

Saved:
  cleaned_kpi_data.csv - 397,884 rows (Tableau source)
  ../outputs/kpi_summary.csv - overall KPI totals
  ../outputs/monthly_kpi.csv - 13 months
  ../outputs/category_kpi.csv - 8 categories
  ../outputs/category_performance.csv - 7 categories (excl. other)


## 7. Visualization & Key Insights

Visualize KPI results and summarize important findings.